DS 5001 - Exploratory Text Analytics Final Project. 

Haejin Son (umw7eg). 

April 23, 2026. 

Digital critical edition of a corpus about Korea from Project Gutenberg via following operations: 

1. Convert the collection from source foramts (F0) into a set of tables that conform to the Standard Text Analytic Data Model (F2) and to the ML Corpus Format (F1).  

2. Annotate the set of tables with statistical and linguistic features using NLP libraries, e.g. NLTK (F3).  

3. Create a vector representation of the corpus to generate TFIDF values to add to the TOKEN and VOCAB tables (F4). 

4. Extend the annotated and vectorized model with tables and features derived from unsupervised methods such as PCA, LDA, and word2vec (F5).

5. Create a visualization and explore the results. 

6. Write a report to present findings and conclusions about cultural patterns observed in the corpus. 

Deliverables: 

1. A jupyter notebook with the code 

2. Manifest file with provenance (source and URLs), location (link to the source in UVA Box), description (general subject matter; Outsiders' view of Korea), and format (description of the file formats, CSV, etc and internal structure if applicable)

3. Data files including the F2 through F5 data extracted from the corpus, e.g. LIBRARY.csv, TOKEN.csv, VOCAB.csv, and the following tables with appropriate index such as OCHO index: 

- Principal Components (PCA)
- Topic Models (LDA)
- Word Embeddings (word2vec)
- Sentiment Analysis

In [1]:
import pandas as pd
import numpy as np

from glob import glob 

import re #regular expression
import nltk #natural language toolkit
import requests #to download the text files

import os, re

from nltk.tokenize import sent_tokenize, word_tokenize, WhitespaceTokenizer
from nltk import pos_tag
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation as LDA, PCA
from sklearn.preprocessing import StandardScaler
from gensim.models import Word2Vec
from scipy.linalg import eigh

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

%matplotlib inline

nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('stopwords')
nltk.download('tagsets')

[nltk_data] Downloading package punkt to /Users/haejin/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/haejin/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/haejin/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package tagsets to /Users/haejin/nltk_data...
[nltk_data]   Package tagsets is already up-to-date!


True

In [2]:
OHCO = ['book_id', 'chap_num', 'para_num', 'sent_num', 'token_num']

In [3]:
# I used Claude to search for and select various books about 
# outsider's view of Korea in 19th-20th century from Project Gutenberg website. 
# and a general description of the books as metadata for LIBRARY table

LIBRARY = [
    {'book_id': 67141, 'author': 'Griffis', 'title': 'Corea The Hermit Nation',   'year': 1882, 'nationality': 'American', 'visited': False,  'stance': 'exotic-neutral'},
    {'book_id': 69300, 'author': 'Bishop',  'title': 'Korea and Her Neighbors',    'year': 1897, 'nationality': 'British',  'visited': True,   'stance': 'sympathetic-traveler'},
    {'book_id': 52127, 'author': 'Hulbert', 'title': 'History of Korea Vol 1',     'year': 1905, 'nationality': 'American', 'visited': True,   'stance': 'pro-korean'},
    {'book_id': 73071, 'author': 'Ladd',    'title': 'In Korea with Marquis Ito',  'year': 1908, 'nationality': 'American', 'visited': True,   'stance': 'pro-japanese'},
]

LIB = pd.DataFrame(LIBRARY).set_index('book_id')
LIB


,author,title,year,nationality,visited,stance
book_id,,,,,,
67141,Griffis,Corea The Hermit Nation,1882,American,False,exotic-neutral
69300,Bishop,Korea and Her Neighbors,1897,British,True,sympathetic-traveler
52127,Hulbert,History of Korea Vol 1,1905,American,True,pro-korean
73071,Ladd,In Korea with Marquis Ito,1908,American,True,pro-japanese


## F0 (source text) directly pulled from Project Gutenberg using requests library

In [4]:
def download_text(url):
    response = requests.get(url)
    response.raise_for_status()
    return response.text

raw_texts = {}

for book_info in LIBRARY:
    book_id = book_info['book_id']  
    urls = [
        f"https://www.gutenberg.org/files/{book_id}/{book_id}-0.txt",
        f"https://www.gutenberg.org/files/{book_id}/{book_id}.txt", 
        f"https://www.gutenberg.org/cache/epub/{book_id}/pg{book_id}.txt"
    ]
    print(f"Downloading {book_id}...")
    
    for url in urls:
        try:
            print(f"Trying {book_id} with {url}")
            raw_texts[book_id] = download_text(url)

            print(f"  Downloaded {len(raw_texts[book_id])} characters")
            downloaded = True
            break
        
        except:
            continue
    
    if not downloaded:
        print(f"  Failed to download {book_id}")

Trying 67141 with https://www.gutenberg.org/files/67141/67141-0.txt
  Downloaded 1242009 characters
Trying 69300 with https://www.gutenberg.org/files/69300/69300-0.txt
  Downloaded 1002030 characters
Trying 52127 with https://www.gutenberg.org/files/52127/52127-0.txt
  Downloaded 979510 characters
Trying 73071 with https://www.gutenberg.org/files/73071/73071-0.txt
  Downloaded 949537 characters


# Inspect the data files

In [5]:
dfs = {}

for book_id, text in raw_texts.items():
    lines = text.split('\n')
    df = pd.DataFrame(lines, columns=['line_str'])
    df.index.name = 'line_num'
    dfs[book_id] = df
    print(f"{book_id}: {len(df)} lines")

67141: 21254 lines
69300: 18545 lines
52127: 16005 lines
73071: 15245 lines


In [6]:
book_boundaries = {}

for book_id, df in dfs.items():
    print(f"Book: {book_id}")
    
    a = df.line_str.str.match(r"\*\*\*\s*START OF TH(IS|E) PROJECT")
    b = df.line_str.str.match(r"\*\*\*\s*END OF TH(IS|E) PROJECT")
    
    start_idx = a[a].index[0]        # first True in 'a'
    end_idx = b[b].index[0]          # first True in 'b'
    
    print(f"Start line: {start_idx}")
    print(f"End line: {end_idx}")
    print(f"Total content lines: {end_idx - start_idx - 1}")
    
    book_boundaries[book_id] = {
        'start_line': start_idx + 1,
        'end_line': end_idx - 1
    }

#print the first 50 lines of the actual content for each book
    print(f"\nFirst lines of content:")
    book_lines = df.loc[start_idx+1 : start_idx+100, "line_str"]
    for idx, line in book_lines.items():
        print(f"  {idx}: {line[:100]}")
        #below code is to check the chapter markers before defining the patterns

# Text from Griffis:
# I used Claude to find the line 62 in the preface 
# actually corrected the chapter header from "I.—ANCIENT AND MEDIÆVAL HISTORY" 
# to "I. ANCIENT AND MEDIÆVAL HISTORY".. and the preface is very long.. 
# This is one of the changes he made in the final edition of this book! 


Book: 67141
Start line: 0
End line: 21252
Total content lines: 21251

First lines of content:
  1: 
  2:                                  COREA
  3:                            THE HERMIT NATION
  4: 
  5:                    I.—ANCIENT AND MEDIÆVAL HISTORY
  6:                       II.—POLITICAL AND SOCIAL COREA
  7:                           III.—MODERN AND RECENT HISTORY
  8: 
  9: 
  10:                                    BY
  11:                          WILLIAM ELLIOT GRIFFIS
  12:           FORMERLY OF THE IMPERIAL UNIVERSITY OF TOKIO, JAPAN
  13:                     AUTHOR OF “THE MIKADO’S EMPIRE”
  14: 
  15: 
  16:                   NINTH EDITION, REVISED AND ENLARGED
  17: 
  18:                                 NEW YORK
  19:                         CHARLES SCRIBNER’S SONS
  20:                                   1911
  21: 
  22: 
  23: 
  24: 
  25: 
  26: 
  27: 
  28: 
  29:                                    TO
  30: 
  31:                           ALL COREAN PATRIOTS:
 

Only Bishop uses standard chapter pattern: CHAPTER I. All others use their own patterns, so I need to create custom patterns for each book. It is important to note that each corpus has its own unique structure, e.g. parts, stories, sections, so the chapter level varies. I identify the 'chapter' level as 'top-level textual unit' for each corpus. 

In [7]:
roman = r'[IVXLCDM]+'
caps  = r"[A-Z][A-Z\s,'\-:]+"
dash  = r'[—\-]{1,2}'

chap_pats = {
# I used Claude to generate specific regex functions 
# as each book has unique formatting for chapter headers.

    # 67141 — Griffis; does not use chapters; I.—ANCIENT AND MEDIÆVAL HISTORY
    'griffis': {
        'book_id': 67141,
       # 'chapter': re.compile(r'^\s*[IVXLCDM]+\.?\s*[\.\-—–]?\s*[A-Z]')
       # 'chapter': re.compile(r'^\s*[IVXLCDM]+\.?\s*[—\-–]\s*[A-Z]{2,}')
         'chapter': re.compile(r'^\s*CHAPTER\s+[IVXLCDM]+', re.IGNORECASE)
    },

    # 69300 — Bishop; uses standard pattern; CHAPTER I. 
    'bishop': {
        'book_id': 69300,
        'chapter': re.compile(r'^\s*CHAPTER\s+[IVXLCDM]+\.?\s*$', re.IGNORECASE)
    },

    # 52127 — Hulbert; uses title case in caps and numbers; ANCIENT KOREA 2257 B.C.-890 A.D.
    'hulbert': {
        'book_id': 52127,
        'chapter': re.compile(r'^\s*CHAPTER\s+[IVXLCDM]+', re.IGNORECASE)
    },

    # 73071 — Ladd; uses PART I, PART II
    'ladd': {
        'book_id': 73071,
        'chapter': re.compile(r'^\s*CHAPTER\s+[IVXLCDM]+', re.IGNORECASE)
    }
}

## DOC table is created using OHCO index

In [8]:
def acquire_texts(LIB, raw_texts, chap_pats, book_boundaries, OHCO=OHCO):

    my_doc = []

    for book_id in LIB.index:

        print(f"\nProcessing {book_id}...")

        start_idx = book_boundaries[book_id]['start_line']
        end_idx = book_boundaries[book_id]['end_line']
        
        lines = raw_texts[book_id].split('\n')
        lines = lines[start_idx:end_idx] #removed boilerplate

        # Griffis has a lot of preface content that needs to be removed
        if book_id == 67141:
            lines = lines[1000:]
            
        df = pd.DataFrame({'line_str': lines})
        df.index.name = 'line_num'
        
        df['line_str'] = df['line_str'].str.strip()
        df['book_id'] = book_id

        # normalize punctuation
        df['line_str'] = df['line_str'].str.replace('—', ' — ', regex=False)
        df['line_str'] = df['line_str'].str.replace('-', ' - ', regex=False)

        # used Cluade to write code to remove lines that look like table of contents (toc) 
        is_toc_line = df['line_str'].str.contains(r'\.{3,}| \d+$', regex=True)
        df = df[~is_toc_line].copy()

        # Lookup correct pattern
        author = LIB.loc[book_id, 'author'].lower()
        chap_pat = chap_pats[author]['chapter']

        # find chapters
        chap_lines = df.line_str.str.contains(chap_pat, regex=True, na=False)

        print(author, chap_pat)
        print(f"Chapter lines found: {chap_lines.sum()}")

        # chapter numbering
        df['chap_num'] = chap_lines.cumsum()

        # Drop pre-chapter content; title, prefaces, etc
        df = df[df['chap_num'] > 0]

        # Group into chapters
        df = df.groupby(['book_id','chap_num']).line_str.apply('\n'.join).to_frame()

        # Paragraph split
        df = df['line_str'].str.split(r'\n{2,}').explode().to_frame(name='para_str')
        df = df.reset_index()
        df['para_num'] = df.groupby(['book_id','chap_num']).cumcount() + 1
        df = df.set_index(['book_id','chap_num','para_num'])
        
        df.index.names = ['book_id','chap_num','para_num']

        # Clean
        df['para_str'] = df['para_str'].str.replace('\n', ' ', regex=True).str.strip()
        df = df[df['para_str'] != '']

        my_doc.append(df)

    DOC = pd.concat(my_doc)

    return LIB, DOC

LIB, DOC = acquire_texts(LIB, raw_texts, chap_pats, book_boundaries,OHCO)

print(f"\nProcessed {len(LIB)} books")
print(f"Total paragraphs: {len(DOC)}")


Processing 67141...
griffis re.compile('^\\s*CHAPTER\\s+[IVXLCDM]+', re.IGNORECASE)
Chapter lines found: 57

Processing 69300...
bishop re.compile('^\\s*CHAPTER\\s+[IVXLCDM]+\\.?\\s*$', re.IGNORECASE)
Chapter lines found: 37

Processing 52127...
hulbert re.compile('^\\s*CHAPTER\\s+[IVXLCDM]+', re.IGNORECASE)
Chapter lines found: 35

Processing 73071...
ladd re.compile('^\\s*CHAPTER\\s+[IVXLCDM]+', re.IGNORECASE)
Chapter lines found: 20

Processed 4 books
Total paragraphs: 6560


In [9]:
LIB.to_csv('LIB.csv')
DOC.to_csv('DOC.csv')

print("\nSaved: LIB.csv, DOC.csv")
print("proceed to F3 (Annotate Tables with Statistical and Linguistic Features (NLTK))")


Saved: LIB.csv, DOC.csv
proceed to F3 (Annotate Tables with Statistical and Linguistic Features (NLTK))


In [10]:
#DOC.sample(20)

In [11]:
#print(LIB)

## Tokenize DOC table to the token level and annotate each token with POS, stem, stopeword, and punctuation. 

This pipeline follows the template from my module 4 code exercise.

In [12]:
# F3 – Annotate Tables with Statistical and Linguistic Features (NLTK)

for pkg in ['punkt', 'punkt_tab', 'averaged_perceptron_tagger',
            'averaged_perceptron_tagger_eng', 'stopwords']:
    nltk.download(pkg, quiet=True)

def tokenize(doc_df, OHCO=OHCO, remove_pos_tuple=True, ws=False):

    # Paragraphs to sentences
    df = (doc_df['para_str']
          .apply(lambda x: pd.Series(nltk.sent_tokenize(x)))
          .stack()
          .to_frame()
          .rename(columns={0: 'sent_str'}))

    # Inner function renamed to avoid shadowing nltk.word_tokenize
    def _pos_tokenize(x):
        if ws:
            s = pd.Series(nltk.pos_tag(WhitespaceTokenizer().tokenize(x)))
        else:
            s = pd.Series(nltk.pos_tag(nltk.word_tokenize(x)))
        return s

    # Sentences to tokens with inline POS tagging
    df = (df['sent_str']
          .apply(_pos_tokenize)
          .stack()
          .to_frame()
          .rename(columns={0: 'pos_tuple'}))

    # Unpack (word, POS) tuple
    df['pos']       = df['pos_tuple'].apply(lambda x: x[1])
    df['token_str'] = df['pos_tuple'].apply(lambda x: x[0])

    if remove_pos_tuple:
        df = df.drop(columns=['pos_tuple'])

    df.index.names = OHCO
    return df

TOKEN = tokenize(DOC)
print(f"  Total tokens: {len(TOKEN):,}")

# Annotate TOKEN with linguistic features
_stemmer  = PorterStemmer()
_stop_set = set(stopwords.words('english'))

_POS_MAP = {
    'NN':'NOUN', 'NNS':'NOUN', 'NNP':'NOUN',  'NNPS':'NOUN',
    'VB':'VERB', 'VBD':'VERB', 'VBG':'VERB',  'VBN':'VERB',
    'VBP':'VERB','VBZ':'VERB',
    'JJ':'ADJ',  'JJR':'ADJ',  'JJS':'ADJ',
    'RB':'ADV',  'RBR':'ADV',  'RBS':'ADV',
}

TOKEN['token_str_lower'] = TOKEN['token_str'].str.lower()
TOKEN['is_alpha']        = TOKEN['token_str'].str.isalpha()
TOKEN['is_stop']         = TOKEN['token_str_lower'].isin(_stop_set)
TOKEN['is_punct']        = TOKEN['token_str'].apply(lambda x: not any(c.isalnum() for c in x))
TOKEN['stem']            = TOKEN.apply(
    lambda r: _stemmer.stem(r['token_str_lower']) if r['is_alpha'] else r['token_str_lower'],
    axis=1
)
TOKEN['pos_coarse'] = TOKEN['pos'].map(_POS_MAP).fillna('OTHER')

# VOCAB table 

# Restrict to content words: alphabetic and not a stopword
_alpha = TOKEN[TOKEN['is_alpha'] & ~TOKEN['is_stop']].reset_index()

VOCAB = (
    _alpha
    .groupby('stem')
    .agg(
        n_tokens     = ('token_str',  'count'),
        n_docs       = ('book_id',    'nunique'),
        pos_mode     = ('pos',        lambda s: s.mode().iloc[0]),
        pos_coarse   = ('pos_coarse', lambda s: s.mode().iloc[0]),
        example_word = ('token_str',  lambda s: s.value_counts().index[0])
    )
)

total_tokens    = VOCAB['n_tokens'].sum()
n_books         = TOKEN.index.get_level_values('book_id').nunique()
VOCAB['tf']     = VOCAB['n_tokens'] / total_tokens        
# relative term freq
VOCAB['df']     = VOCAB['n_docs']   / n_books             
# doc freq (0–1)
VOCAB           = VOCAB.sort_values('n_tokens', ascending=False)

print(f"  Vocabulary size (content words): {len(VOCAB):,}")

  Total tokens: 793,363
  Vocabulary size (content words): 13,960


In [13]:
print(TOKEN.head(10).to_string())

                                              pos token_str token_str_lower  is_alpha  is_stop  is_punct     stem pos_coarse
book_id chap_num para_num sent_num token_num                                                                                
67141   1        1        0        0           NN   CHAPTER         chapter      True    False     False  chapter       NOUN
                                   1          NNP       LII             lii      True    False     False      lii       NOUN
                                   2            .         .               .     False    False      True        .      OTHER
        2        1        0        0           NN   CHAPTER         chapter      True    False     False  chapter       NOUN
                                   1          NNP      LIII            liii      True    False     False     liii       NOUN
                                   2            .         .               .     False    False      True        .      OTHER


In [14]:
print(VOCAB.head(10).to_string())

         n_tokens  n_docs pos_mode pos_coarse example_word        tf   df
stem                                                                     
japanes      2249       4       JJ        ADJ     Japanese  0.006590  1.0
king         2172       4       NN       NOUN         king  0.006365  1.0
one          2117       4       CD      OTHER          one  0.006204  1.0
korean       1948       4       JJ        ADJ       Korean  0.005708  1.0
korea        1492       4      NNP       NOUN        Korea  0.004372  1.0
time         1329       4       NN       NOUN         time  0.003894  1.0
japan        1295       4      NNP       NOUN        Japan  0.003795  1.0
peopl        1257       4      NNS       NOUN       people  0.003683  1.0
made         1241       4      VBN       VERB         made  0.003637  1.0
year         1233       4       NN       NOUN         year  0.003613  1.0


In [15]:
# POS distribution across books
pos_dist = (
    TOKEN[TOKEN['is_alpha']]
    .groupby(['book_id', 'pos_coarse'])
    .size()
    .unstack(fill_value=0)
)
pos_dist = pos_dist.div(pos_dist.sum(axis=1), axis=0).round(4)
pos_dist = pos_dist.join(LIB[['author', 'stance']])

print("\n── POS distribution (proportion of alpha tokens) ──")
print(pos_dist.to_string())


── POS distribution (proportion of alpha tokens) ──
            ADJ     ADV    NOUN   OTHER    VERB   author                stance
book_id                                                                       
52127    0.0622  0.0409  0.2878  0.4379  0.1713  Hulbert            pro-korean
67141    0.0808  0.0385  0.3010  0.4334  0.1463  Griffis        exotic-neutral
69300    0.0910  0.0417  0.2908  0.4297  0.1469   Bishop  sympathetic-traveler
73071    0.0903  0.0521  0.2665  0.4461  0.1450     Ladd          pro-japanese


Hulbert's book has more verb than the other three authors' books. This was expected as his book is historical and would have more action words. Bishop's book has the most adjective, which is also expected as her book is a travel book and would have more descriptive words. 

In [16]:
# TOKEN table has one row per token, indexed by full OHCO (book/chap/para/sent/token)
# VOCAB table has one row per unique stem with aggregate corpus statistics

TOKEN.to_csv('TOKEN.csv')
VOCAB.to_csv('VOCAB.csv')

print("\nSaved: TOKEN.csv, VOCAB.csv")


Saved: TOKEN.csv, VOCAB.csv


## F4: TF-IDF Vectorisation

create a vector representation of each book to create TFIDF values to add to the TOKEN and VOCAB tables saved above. This section uses the pipeline from my module 5 code exercise, which is TOKEN, BOW, DTCM, TF, IDF, and TFIDF. 

In [17]:
OHCO  = ['book_id', 'chap_num', 'para_num', 'sent_num', 'token_num']
BOOKS = OHCO[:1]
CHAPS = OHCO[:2]
PARAS = OHCO[:3]
SENTS = OHCO[:4]

In [18]:
# Config for book level bag to compare the four authors; for document level comparisons and analysis

bag          = BOOKS        
count_method = 'n'           
tf_method    = 'sum'          
tf_norm_k    = 0.5           
idf_method   = 'standard'    

# First, add term_id to TOKEN 
# VOCAB is indexed by stem; I need an integer term_id for BOW groupby.
# If term_id does not already exist in VOCAB, create it here.

if 'term_id' not in VOCAB.columns:
    VOCAB = VOCAB.reset_index()           # stem becomes a column
    VOCAB.index.name = 'term_id'
    VOCAB = VOCAB.reset_index()           # term_id is now a column
    VOCAB = VOCAB.set_index('stem')

# Map stem: term_id onto TOKEN
TOKEN['term_id'] = TOKEN['stem'].map(VOCAB['term_id'])

print("TOKEN with term_id:")
print(TOKEN[['token_str', 'stem', 'pos', 'term_id']].head(10).to_string())


# Build BOW  (Bag of Words)
# Group by chosen bag level + term_id → raw count (n) and binary count (c)

BOW = (
    TOKEN
    .dropna(subset=['term_id'])
    .groupby(bag + ['term_id'])['term_id']
    .count()
    .to_frame()
    .rename(columns={'term_id': 'n'})
)

BOW['c'] = BOW['n'].astype('bool').astype('int')   # binary: 1 if present, 0 if not

print(f"  BOW shape: {BOW.shape}")
print(BOW.head(10).to_string())

# DTCM (Document-Term Count Matrix)
# Unstack term_id to columns; rows = documents
DTCM = BOW[count_method].unstack(fill_value=0).astype('int')

print(f"  DTCM shape: {DTCM.shape}  (documents × terms)")
print(DTCM.iloc[:5, :5].to_string())

# TF: Term Frequency
if tf_method == 'sum':
    TF = DTCM.T / DTCM.T.sum()                          # proportion of doc total
elif tf_method == 'max':
    TF = DTCM.T / DTCM.T.max()                          # scale by doc max
elif tf_method == 'log':
    TF = np.log10(1 + DTCM.T)                           # log-smoothed
elif tf_method == 'raw':
    TF = DTCM.T.copy()                                   # unmodified counts
elif tf_method == 'double_norm':
    TF = DTCM.T / DTCM.T.max()
    TF = tf_norm_k + (1 - tf_norm_k) * TF               # augmented normalisation
elif tf_method == 'binary':
    TF = DTCM.T.astype('bool').astype('int')             # presence only

TF = TF.T      # back to documents × terms orientation

# DF and IDF 
# Document Frequency, Inverse DF

DF = DTCM[DTCM > 0].count()      # number of docs each term appears in
N  = DTCM.shape[0]                # total number of documents

print(f"  N documents: {N}")

print(f"\nComputing IDF using idf_method='{idf_method}' …")

if idf_method == 'standard':
    IDF = np.log10(N / DF)
elif idf_method == 'max':
    IDF = np.log10(DF.max() / DF)
elif idf_method == 'smooth':
    IDF = np.log10((1 + N) / (1 + DF)) + 1

# TFIDF 

TFIDF = TF * IDF

print(TFIDF.iloc[:5, :5].to_string())

TOKEN with term_id:
                                             token_str     stem  pos  term_id
book_id chap_num para_num sent_num token_num                                 
67141   1        1        0        0           CHAPTER  chapter   NN    337.0
                                   1               LII      lii  NNP   8836.0
                                   2                 .        .    .      NaN
        2        1        0        0           CHAPTER  chapter   NN    337.0
                                   1              LIII     liii  NNP   8837.0
                                   2                 .        .    .      NaN
        3        1        0        0           CHAPTER  chapter   NN    337.0
                                   1               LIV      liv  NNP   8874.0
                                   2                 .        .    .      NaN
                 2        0        0              LIST     list   NN   1390.0
  BOW shape: (29998, 2)
                    

In [19]:
# merge TFIDF results back to BOW, VOCAB, and TOKEN for different levels of text analysis 

# Build id_to_word lookup
# TFIDF.columns are term_id floats; VOCAB is indexed by stem.
# Build reverse map: term_id (float) to example_word

id_to_word = (
    VOCAB.reset_index()
         .set_index('term_id')
         ['example_word']
         .to_dict()
)

# Add tf and tfidf back to BOW
BOW['tf']    = TF.stack()
BOW['tfidf'] = TFIDF.stack()

# Write DF, IDF, and TFIDF aggregates to VOCAB
# TFIDF.columns = term_id (float); VOCAB.index = stem
# Bridge term_id to stem, then assign into VOCAB

tid_to_stem = (
    VOCAB.reset_index()
         .set_index('term_id')
         ['stem']
)

tfidf_agg = pd.DataFrame({
    'df'          : DF,
    'idf'         : IDF,
    'tfidf_mean'  : TFIDF[TFIDF > 0].mean(),
    'tfidf_sum'   : TFIDF.sum(),
    'tfidf_median': TFIDF[TFIDF > 0].median(),
    'tfidf_max'   : TFIDF.max(),
})
tfidf_agg.index.name = 'term_id'
tfidf_agg['stem']    = tfidf_agg.index.map(tid_to_stem)
tfidf_agg            = tfidf_agg.dropna(subset=['stem']).set_index('stem')

for col in ['df', 'idf', 'tfidf_mean', 'tfidf_sum', 'tfidf_median', 'tfidf_max']:
    VOCAB[col] = tfidf_agg[col]

if 'tfidf' in TOKEN.columns:
    TOKEN = TOKEN.drop(columns=['tfidf'])

# Join tfidf back onto TOKEN
# TOKEN is at token level; tfidf lives at (bag + term_id) level in BOW.
# Merge so each token gets the tfidf weight of its term in its document.

TOKEN = (
    TOKEN.reset_index()
         .merge(
             BOW[['tfidf']].reset_index(),
             on  = bag + ['term_id'],
             how = 'left'
         )
         .set_index(OHCO)
)
TOKEN['tfidf'] = TOKEN['tfidf'].fillna(0.0)

This small reduction is expected as the Project Gutenberg corpus are quite standardized. Also, my preprocessing pipeline already removed noise. 

In [20]:
# Top 10 terms by tfidf_sum

top10 = (
    VOCAB
    .sort_values('tfidf_sum', ascending=False)
    .head(10)
    [['example_word', 'pos_coarse', 'n_tokens', 'df',
      'idf', 'tfidf_mean', 'tfidf_sum', 'tfidf_max']]
)
print(top10.to_string())

for book_id in LIB.index:
    #get book metadata for author name and their stance for Korea
    author = LIB.loc[book_id, 'author']
    stance = LIB.loc[book_id, 'stance']

    book_rows = TFIDF.loc[[book_id]]  
    top = (
        book_rows.mean() 
                 .sort_values(ascending=False) # highest to lowest
                 .head(10)
    )
    top.index = top.index.map(lambda tid: id_to_word.get(tid, str(tid)))

    print(f"\n{author} ({stance}):")
    print(top.round(6).to_string())

       example_word pos_coarse  n_tokens  df      idf  tfidf_mean  tfidf_sum  tfidf_max
stem                                                                                   
corean       Corean       NOUN       948   1  0.60206    0.004351   0.004351   0.004351
corea         Corea       NOUN       601   1  0.60206    0.002759   0.002759   0.002759
koryŭ         Koryŭ       NOUN       436   1  0.60206    0.002431   0.002431   0.002431
sil             Sil       NOUN       419   1  0.60206    0.002336   0.002336   0.002336
gu               gu       NOUN       332   1  0.60206    0.001851   0.001851   0.001851
ryŭ             ryŭ       NOUN       317   1  0.60206    0.001767   0.001767   0.001767
păk             Păk       NOUN       217   1  0.60206    0.001210   0.001210   0.001210
dæmon         dæmon       NOUN       164   1  0.60206    0.001026   0.001026   0.001026
yŭng           yŭng      OTHER       179   1  0.60206    0.000998   0.000998   0.000998
chō             Chō       NOUN  

TFIDF words show the most distinctive (unique) terms for each book, so commonly used words in one specific book will not appear in the TFIDF top 10 lists. 

Hulbert's top 10 terms are all Korean words in romanized form (Koryŭ, Sil, gu, ryŭ, Păk, yŭng, Ch, P, wŭn, je). This shows that Hulbert in some extent knew the Korean langauge and was deeply immersed in Korean culture. 

Griffis (1882) seems to use a lot of ancient names (Corean) and kingdoms (Shinra, Korai, Manchiu) compared to other authors. He focuses more on historical context and kingdom names. Expected as he is a historian writing about Korea. 

Ladd (1908) uses Ito and Christian frequently, which is expected as he is a missionary writing about Korea and pro-Japanese. Ito is a Japanese authority figure. Also he seems to write about geopolitical context of Korea as he mentions, Russia, Hague (Hague convention for Korean diplomacy?), Cabinet and Suwon (city in Korea).  

In [21]:
TOKEN.to_csv('TOKEN.csv')
VOCAB.to_csv('VOCAB.csv')
BOW.to_csv('BOW.csv')
DTCM.to_csv('DTCM.csv')

print("\nSaved: TOKEN.csv, VOCAB.csv, BOW.csv, DTCM.csv")


Saved: TOKEN.csv, VOCAB.csv, BOW.csv, DTCM.csv


## F5: PCA, LDA, and Word2Vec

PCA (Principal Component Analysis): Following the module 7 code exercise approach, I will use eigh() on a term covariance matrix to perform PCA and project to a document-component table (DCM).

In [22]:
#BOW at book level
BOW = TOKEN.groupby(['book_id', 'term_id']).size().to_frame().rename(columns={0: 'n'})

#DTM
DTM = BOW['n'].unstack(fill_value=0)
print(f"DTM shape: {DTM.shape}")

# create a TFIDF matrix
def get_tfidf(dtm):
    #term frequency: normalize each row by total tokens in that document
    tf    = dtm.apply(lambda row: row / row.sum(), axis=1)
    #document frequency: count how many documents each term appears in
    n_docs = dtm.shape[0]
    df    = dtm.gt(0).sum()
    #inverse document frequency: penalize terms that appear in many documents
    idf   = np.log2(n_docs / df)
    
    return tf * idf

TFIDF_pca = get_tfidf(DTM)

# feature reduction by removing proper nouns and keeping only top 4000 terms by tfidf_sum
VOCAB_FILTERED = VOCAB[~VOCAB['pos_mode'].isin(['NNP', 'NNPS'])]

# term_id is a column in VOCAB; tfidf_sum already computed in F4
top_4000_terms = VOCAB_FILTERED['tfidf_sum'].sort_values(ascending=False).head(4000).index

# TFIDF_pca columns are term_ids (floats); map stems → term_ids
stem_to_tid = VOCAB.reset_index().set_index('stem')['term_id']
top_tids    = stem_to_tid.loc[top_4000_terms].dropna().values

# Keep only term_ids that actually exist as columns
top_tids    = [t for t in top_tids if t in TFIDF_pca.columns]
TFIDF_pca   = TFIDF_pca[top_tids]
print(f"TFIDF after feature reduction: {TFIDF_pca.shape}")

# L2 normalization and centering
TFIDF_pca = TFIDF_pca.apply(lambda x: x / np.sqrt(np.square(x).sum()), axis=1)
TFIDF_pca = TFIDF_pca - TFIDF_pca.mean()

# Term covariance matrix (COV). Apply eigendecomposition to the COV table.
COV = TFIDF_pca.T.dot(TFIDF_pca) / (TFIDF_pca.shape[0] - 1)
print(f"COV shape: {COV.shape}")

%time eig_vals, eig_vecs = eigh(COV)

TERM_IDX = COV.index

EIG_VAL = pd.DataFrame(eig_vals,  index=TERM_IDX, columns=['eig_val'])
EIG_VAL.index.name = 'term_id'

EIG_VEC = pd.DataFrame(eig_vecs,  index=TERM_IDX, columns=TERM_IDX)

# Combine eigenvalues and eigenvectors
EIG_PAIRS = EIG_VAL.join(EIG_VEC.T)

#Add explained variance column and run the top 10 components
EIG_PAIRS['exp_var'] = np.round(
    (EIG_PAIRS['eig_val'] / EIG_PAIRS['eig_val'].sum()) * 100, 2
)
EIG_PAIRS = EIG_PAIRS.sort_values('eig_val', ascending=False)

print("\nTop 10 components by explained variance:")
print(EIG_PAIRS[['eig_val', 'exp_var']].head(10).to_string())

COMPS = EIG_PAIRS.sort_values('exp_var', ascending=False).head(10).reset_index(drop=True)
COMPS.index = [f'PC{i}' for i in COMPS.index.tolist()]
COMPS.index.name = 'comp_id'

# DCM: project books onto component space
DCM = TFIDF_pca.dot(COMPS[TERM_IDX].T)
DCM.index.name = 'book_id'

# Join author and stance metadata from LIB table
DCM = DCM.reset_index().merge(LIB[['author', 'stance', 'nationality', 'visited']], on='book_id')

print("\nDCM:")
print(DCM[['book_id', 'author', 'stance', 'PC0', 'PC1', 'PC2']].to_string())

DTM shape: (4, 13960)
TFIDF after feature reduction: (4, 4000)
COV shape: (4000, 4000)
CPU times: user 57.7 s, sys: 5.27 s, total: 1min 2s
Wall time: 10.1 s

Top 10 components by explained variance:
              eig_val  exp_var
term_id                       
5168.0   3.426081e-01    36.04
5023.0   3.080508e-01    32.40
4966.0   2.999948e-01    31.56
4963.0   6.236672e-16     0.00
4964.0   4.935647e-16     0.00
5225.0   3.943202e-16     0.00
5150.0   3.832445e-16     0.00
4686.0   3.247627e-16     0.00
4834.0   2.441177e-16     0.00
4739.0   1.693267e-16     0.00

DCM:
   book_id   author                stance       PC0       PC1       PC2
0    52127  Hulbert            pro-korean -0.876918 -0.029612 -0.028202
1    67141  Griffis        exotic-neutral  0.253905  0.727521  0.321082
2    69300   Bishop  sympathetic-traveler  0.299267 -0.623227  0.467240
3    73071     Ladd          pro-japanese  0.323745 -0.074682 -0.760120


In [23]:
# plots for comparison with metadata
def vis_pcs(M, a, b, label='author', prefix='PC'):
    fig = px.scatter(
        M, x=prefix+str(a), y=prefix+str(b),
        color=label,
        hover_name='author',
        hover_data=['stance'],
        text='author',
        title=f'{prefix}{a} vs {prefix}{b}  (colored by {label})'
    )
    fig.show()

vis_pcs(DCM, 0, 1, label='stance')
vis_pcs(DCM, 1, 2, label='stance')

# Top positive and negative terms per component
tid_to_word = VOCAB.reset_index().set_index('term_id')['example_word'].to_dict()

for pc in ['PC0', 'PC1', 'PC2']:
    loading_series = COMPS.loc[pc, TERM_IDX]
    top_pos = loading_series.nlargest(10)
    top_neg = loading_series.nsmallest(10)

    top_pos.index = top_pos.index.map(lambda t: tid_to_word.get(t, str(t)))
    top_neg.index = top_neg.index.map(lambda t: tid_to_word.get(t, str(t)))

    print(f"\n{pc} positive loadings")
    print(top_pos.round(5).to_string())
    print(f"{pc} negative loadings")
    print(top_neg.round(5).to_string())


PC0 positive loadings
term_id
dæmon         0.22222
sen           0.17212
Russian       0.11561
mikado        0.09552
jinrikisha    0.07801
chiu          0.06759
redemption    0.06570
chiefly       0.05990
church        0.05532
castle        0.05437
PC0 negative loadings
term_id
gu      -0.49380
ryŭ     -0.47149
yŭng    -0.26624
wŭn     -0.16510
je      -0.15750
sŭng    -0.13089
ung     -0.12791
hă      -0.11453
gön     -0.11006
gyŭng   -0.08627

PC1 positive loadings
term_id
sen       0.54852
mikado    0.30441
chiu      0.21539
castle    0.15206
wen       0.08182
French    0.07240
dō        0.06892
bonzes    0.06031
daimiō    0.05456
serfs     0.05169
PC1 negative loadings
term_id
dæmon      -0.51469
döng       -0.09729
exorcism   -0.07846
ren        -0.06277
bulls      -0.05035
pedlars    -0.04708
solitary   -0.04394
Russian    -0.04178
foam       -0.04080
ofttimes   -0.03766

PC2 positive loadings
term_id
dæmon       0.39623
sen         0.24858
mikado      0.13796
chiu        0.097

The first plot shows that Hulbert is separated from the rest of the authors along the x-axis of PC0, which suggests that his writing style is unique, compared to the other three authors. This is supported by the strong negative PC0 result for Hulbert. It is interesting to note that he has lived in korea for a long time as an educator, and later eventually buried in Korea at this request. He uses more Korean historical terms and native perspectives which sets him apart from the other authors. 

Other than that, I don't see a clear pattern or any clustering based on authors or stance on Korea. 

PO0 loadings show clear divide in the corpus about Korea: the positive loadings show Western terms and concepts such as mikado (Japanese term for Emperor), Russian, and church, while the negative loadings show native Korean words in romanization such as gu, wun, and gyung. This shows that the corpus is divided between Western perspective of Korea through foriegn cultural lens whether postiive or negative, and Korean perspectives from rather Korean 'insider' perspective, whether it may be positive or negative. 

Ladd has the highest PO0 score, which suggests that his writing is more focused on Western perspectives. It is interesting to note that despite having pro-Japanese stance, Ladd's text is actually more 'Western' than other authors. Hulbert scored the lowest PO0 score, which suggests that his writing is more focused on native Korean terms. This suggests that he was more focused on Korean perspectives than Western perspectives. It is interesting to note that he later became a Korean independence activist and wrote extensively about Korean independence. 

This shows how colonial-era Western perspectives tended to write about Korea through their own cultural and political lenses, with Hulbert being the exception who engaged with Korean concepts on their own terms. PCA has uncovered the linguistic foundations of how these Western writers constructed their understanding of Korea.

In [24]:
PCA_DCM = DCM.set_index('book_id')
PCA_DCM.to_csv('PCA_DCM.csv')

COMPS.to_csv('PCA_COMPS.csv')
print("\nSaved: PCA_DCM.csv, PCA_COMPS.csv")


Saved: PCA_DCM.csv, PCA_COMPS.csv


LDA (Latent Dirichlet Allocation) - unsupervised topic modeling algorithm that identifies hidden topics in the corpus, under the assumption that documents are mixtures of topics and topics are mixtures of words. 

In [31]:
# Build a document-level text corpus from TOKEN
doc_texts = {}
for book_id in LIB.index:
    words = TOKEN.xs(book_id, level='book_id')
    words = words[words['is_alpha'] & ~words['is_stop']]
    doc_texts[book_id] = ' '.join(words['token_str_lower'].tolist())

docs = [doc_texts[b] for b in LIB.index]
book_ids = list(LIB.index)

# Fit CountVectorizer 
# LDA works on raw counts
cv = CountVectorizer(max_features=5000, min_df=1)
DTM_lda = cv.fit_transform(docs)
vocab_lda = cv.get_feature_names_out()

# Fit LDA
# choice of 5 topics because it is a good balance between specificity and generality
N_TOPICS = 5 
lda_model = LDA(n_components=N_TOPICS, random_state=42, max_iter=20)
lda_model.fit(DTM_lda)

# Document-Topic Matrix (THETA)
# how much each book belongs to each topic
THETA = pd.DataFrame(
    lda_model.transform(DTM_lda),
    index=book_ids,
    columns=[f'topic_{i}' for i in range(N_TOPICS)]
)
THETA.index.name = 'book_id'
THETA = THETA.join(LIB[['author', 'stance']])

print("Document-Topic Matrix (THETA):")
print(THETA.to_string())

# Top terms per topic (PHI)
# which words define each topic
PHI = pd.DataFrame(
    lda_model.components_,
    columns=vocab_lda,
    index=[f'topic_{i}' for i in range(N_TOPICS)]
)
PHI.index.name = 'topic_id'

TOP_N = 10
print("\nTop terms per topic:")
for topic in PHI.index:
    top_terms = PHI.loc[topic].nlargest(TOP_N).index.tolist()
    print(f"  {topic}: {', '.join(top_terms)}")

Document-Topic Matrix (THETA):
          topic_0   topic_1   topic_2   topic_3   topic_4   author                stance
book_id                                                                                 
67141    0.001121  0.001785  0.000002  0.000002  0.997089  Griffis        exotic-neutral
69300    0.997824  0.002116  0.000003  0.000003  0.000054   Bishop  sympathetic-traveler
52127    0.000003  0.000003  0.000003  0.999989  0.000003  Hulbert            pro-korean
73071    0.000141  0.999850  0.000003  0.000003  0.000003     Ladd          pro-japanese

Top terms per topic:
  topic_0: korean, one, korea, japanese, two, men, great, seoul, much, chinese
  topic_1: korea, korean, japanese, japan, one, government, foreign, upon, time, would
  topic_2: marquis, sincerity, prejudice, circles, descend, sections, halted, chon, community, hamlets
  topic_3: king, sent, one, people, japanese, emperor, koryŭ, la, sil, made
  topic_4: japanese, corean, one, corea, chinese, japan, china, two,

Ladd's book is heavily focused on the topic 1 which is about Korea, Japan, and government. This says something about his stance on Korea. 

In [26]:
# Save LDA results
THETA.to_csv('LDA_THETA.csv')
PHI.to_csv('LDA_PHI.csv')
print("\nSaved: LDA_THETA.csv, LDA_PHI.csv")


Saved: LDA_THETA.csv, LDA_PHI.csv


Word2Vec (Word Embeddings) learns a list of vectors for every word, so words that appear in similar contexts end up close together in this vector space. By sliding a list of words across the book and training a neural network to predict the surrounding words, it captures semantic relationships between words. 

In [35]:
# Build sentences: list of token lists per sentence
sentences = []
for (book_id, chap, para, sent), grp in TOKEN.groupby(level=['book_id','chap_num','para_num','sent_num']):
    words = grp[grp['is_alpha'] & ~grp['is_stop']]['token_str_lower'].tolist()
    if words:
        sentences.append(words)

print(f"Total sentences for Word2Vec: {len(sentences):,}")

# Train
w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    seed=42,
    epochs=10
)

# Save word vectors as a table
words_in_model = list(w2v_model.wv.key_to_index.keys())
W2V = pd.DataFrame(
    [w2v_model.wv[w] for w in words_in_model],
    index=words_in_model,
    columns=[f'dim_{i}' for i in range(w2v_model.vector_size)]
)
W2V.index.name = 'word'

print(f"Word2Vec vocabulary: {len(W2V):,} words")

# Probe: words most similar to key cultural terms
probes = ['korea', 'japan', 'chinese', 'america', 'colony', 'church']
for term in probes:
    if term in w2v_model.wv:
        similar = w2v_model.wv.most_similar(term, topn=5)
        print(f"\n  '{term}' → {[w for w,_ in similar]}")

Total sentences for Word2Vec: 26,789


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Word2Vec vocabulary: 8,871 words

  'korea' → ['russia', 'corea', 'policy', 'relations', 'peace']

  'japan' → ['russia', 'corea', 'peace', 'treaty', 'china']

  'chinese' → ['japanese', 'army', 'invasion', 'forces', 'force']

  'america' → ['india', 'paramount', 'vital', 'neutrality', 'steadily']

  'colony' → ['settlers', 'steamers', 'civilians', 'colonists', 'communication']

  'church' → ['missions', 'methodist', 'catholic', 'roman', 'protestant']


As we can see from the Word2Vec results, Korea and its geopolitical context were consistently present through language by the four authors during the 19th and 20th centuries. The word Korea clusters with words like Russia, policy, relations, and peace, which suggests that Korea was primarily portrayed as a geopolitical objective. The word, Japan appears near Corea, peace, treaty, and China, which reflects the geopolitical tensions between these countries during this time period. 

The word Chinese appear in a cluster with words such as Japanese, army, invasion, forces which reflects the military conflicts during this time period, in the perspetives of the four authors. It seems the four authors collectively frame China as a military actor. On the other hand, the word America clusters with more abstract and somewhat neutral words such as India, paramount, vital, neutrality, and steadily. This suggests that America's role in the Northeast Asia was portrayed as a strategic and diplomatic actor rather than a military actor by the authors. Specifically, the word paramount and vital suggest America's strategic important in the region, although military terms such as army, forces, invasion do not appear in the same cluster with America. 

Overall, it is interesting to identify Korea's neighboring countries, Japan and China and its ally, America are all in the semantic neighborhoods defined by power, military forces and diplomacy. 

Besides the geopolitical narrative, the word church appears cleanly with missions, methodist, catholic, roman and protestant. This shows that the religious words were self-contained and clearly separated from the political vocabulary. 

In [28]:
# Save Word2Vec embeddings and model
W2V.to_csv('W2V_EMBEDDINGS.csv')
w2v_model.save('w2v_model.bin')
print("\nSaved: W2V_EMBEDDINGS.csv, w2v_model.bin")


Saved: W2V_EMBEDDINGS.csv, w2v_model.bin
